# Market Map: out-of-sample test of the divergence lists

`market_map.ipynb` builds two lists from one 2-year window. This notebook checks whether they mean anything *after* that window:
- **Lookalikes that trade apart** (top 1% business similarity, lowest residual correlation). The claim to test is that these are mispricings or pair trades.
- **Hidden twins** (top 1% residual correlation, least similar businesses). The claim to test is that they share a real exposure.

**Design.** 10 years of prices, cut into rolling folds. Each fold builds the lists on a 2-year *formation* window (the same window the app uses), then measures them on the *next* year, which the lists never saw. The test years don't overlap. Statistics are computed per fold, then averaged across folds with a t-stat on the fold means (Fama-MacBeth). That's because the same stock appears in many pairs, so pair-level t-stats would be badly overstated.

**The trap this avoids.** If you pick pairs *because* their correlation was unusually low, it will rise next year anyway: part of what made it low was noise, and the noise doesn't repeat (regression to the mean). So "the apart pairs converged" proves nothing on its own. Each apart pair is compared with **control pairs that had the same formation correlation but dissimilar businesses**. Only a rebound *beyond* those controls counts as information.

## Pass/fail criteria (written before any results)
- **H1, information:** the apart pairs' next-year residual correlation beats the matched controls. PASS if the mean excess is > 0 and its fold-level t-stat clears the two-sided 5% critical value.
- **H2, tradeable:** in each apart pair, go long the formation-window laggard and short the leader. Hedge with *formation-window* betas (no look-ahead) and hold for the test year. PASS if the mean return is > 0 and significant **and** beats the same trade on all top-1%-similar pairs. Without that second condition, it's just a generic catch-up effect that the list adds nothing to.
- **Hidden twins (descriptive):** the share of their excess correlation that survives into the next year, compared with "obvious twins" (top-correlated pairs with the *most* similar businesses).

## Biases that remain (all of them make a PASS look better than it is)
- **Survivorship:** the universe is today's S&P 500. Laggards that went bust or got dropped are missing, which inflates any catch-up trade.
- **Today's descriptions:** the business text was written after the fact. A company that pivoted during a test year already reads like its new peers.
- **Low power:** 7 folds. A FAIL is fairly informative. A marginal PASS isn't.

## 0. Setup

In [1]:
%pip install -q yfinance sentence-transformers lxml
import json, os, re, sys, time, warnings
from io import StringIO
import numpy as np, pandas as pd, requests, yfinance as yf
from scipy import stats

HISTORY = "10y"
FORM, TEST = 504, 252         # formation = 2 trading years (as in the app), test = the following year
TOP, N_LIST = 0.99, 25        # same list definitions as market_map.ipynb
BIN, MIN_CTRL = 0.02, 20      # controls are matched within 0.02-wide formation-correlation bins, needing at least 20 pairs per bin
CACHE = "desc_cache.json"     # reuse the cache from market_map.ipynb (upload it to Colab)
IN_COLAB = "google.colab" in sys.modules
pd.set_option("display.width", 180)

## 1. Universe and 10 years of prices
The universe and download code are the same as the main notebook. Stocks aren't dropped globally here. Each fold uses the stocks with complete data across its own 3 years.

In [2]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                    headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
meta = pd.read_html(StringIO(html))[0][["Symbol", "Security", "GICS Sector", "GICS Sub-Industry"]]
meta["Symbol"] = meta["Symbol"].str.replace(".", "-", regex=False)
meta = meta.set_index("Symbol")

def download(chunk, tries=3):
    for k in range(tries):
        try:
            df = yf.download(chunk, period=HISTORY, auto_adjust=True, progress=False)["Close"]
            if not df.dropna(how="all").empty:
                return df
        except Exception as e:
            print(f"chunk starting {chunk[0]}: {type(e).__name__}, retrying")
        time.sleep(10 * (k + 1))
    raise RuntimeError(f"price download failed for chunk starting {chunk[0]}")

tickers = list(meta.index) + ["SPY"]
px = pd.concat([download(tickers[i:i + 100]) for i in range(0, len(tickers), 100)], axis=1).ffill(limit=3)
lr = np.log(px).diff().iloc[1:]
mkt = lr.pop("SPY")
assert mkt.notna().all(), "SPY has gaps"
print(f"{lr.shape[1]} stocks, {len(lr)} days, {lr.index[0].date()} to {lr.index[-1].date()}")

503 stocks, 2511 days, 2016-11-22 to 2026-11-18


## 2. Business similarity
The pipeline is identical to `market_map.ipynb`: descriptions (cached), names and boilerplate stripped, MiniLM embeddings, mean-centred, cosine similarity.

In [3]:
from sentence_transformers import SentenceTransformer

def fetch_desc(t, tries=3):
    for k in range(tries):
        try:
            return yf.Ticker(t).info.get("longBusinessSummary") or ""
        except Exception as e:
            if k == tries - 1:
                print(f"\n{t}: {type(e).__name__}, will retry next run"); return None
            time.sleep(5 * 3 ** k)

desc = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
todo = [t for t in lr.columns if desc.get(t) is None]
print(f"fetching {len(todo)} descriptions ({len(lr.columns) - len(todo)} cached)")
for n, t in enumerate(todo, 1):
    desc[t] = fetch_desc(t); time.sleep(0.2)
    if n % 50 == 0 or n == len(todo):
        with open(CACHE, "w") as f: json.dump(desc, f)

BOILER = re.compile(r"[^.]*\b(incorporated|founded|headquartered|formerly known as|changed its name)\b[^.]*\.\s*", re.I)
SUFFIX = re.compile(r"\s*\(.*?\)|,?\s+(Inc\.?|Corp\.?|Corporation|Company|Co\.|plc|Ltd\.?|Limited|Holdings?|Group|N\.V\.|The)$", re.I)
def core_name(n):
    prev = None
    while prev != n: prev, n = n, SUFFIX.sub("", n).strip(" ,")
    return n
def clean(t, n):
    t, c = BOILER.sub("", t), core_name(n)
    for w in sorted({c, c.split()[0]}, key=len, reverse=True):
        if len(w) >= 3: t = re.sub(rf"\b{re.escape(w)}\b", "The company", t)
    return t

names = list(lr.columns)
text = [clean(desc[t], meta.loc[t, "Security"]) if desc.get(t)
        else f"{meta.loc[t, 'Security']}: {meta.loc[t, 'GICS Sub-Industry']}" for t in names]
emb = SentenceTransformer("all-MiniLM-L6-v2", device="cpu").encode(text, normalize_embeddings=True,
                                                                  show_progress_bar=False).astype(np.float64)
emb -= emb.mean(0); emb /= np.linalg.norm(emb, axis=1, keepdims=True)
S, pos = emb @ emb.T, {t: i for i, t in enumerate(names)}
print(f"similarity matrix for {len(names)} stocks")

fetching 0 descriptions (503 cached)


similarity matrix for 503 stocks


## 3. The test
For each fold:
1. Residual returns: regress on SPY separately in the formation and test windows, then compute the residual correlation in each (`F` and `T`).
2. **Information:** each apart pair's `T` minus the mean `T` of dissimilar-business pairs (below-median similarity) that had the same formation correlation.
3. **Trade:** go long the pair's formation-window residual laggard and short its leader, hedged with formation betas, and measure the summed log return over the test year.
4. **Context:** a regression across *all* pairs, `T ~ F + similarity`. It asks whether business similarity predicts next year's correlation once this year's is known. The coefficient is per 1 SD of similarity.

In [4]:
def resid_corr(R, m):
    X = np.column_stack([np.ones(len(m)), m.values])
    coef, *_ = np.linalg.lstsq(X, R.values, rcond=None)
    E = R.values - X @ coef
    return np.corrcoef(E, rowvar=False), coef[1], E

def make_folds(n, form, test):
    out, end = [], n
    while end - test - form >= 0:
        out.append((end - test - form, end - test, end)); end -= test
    return out[::-1]

def run(form=FORM, test=TEST):
    rows, detail = [], None
    for f0, t0, t1 in make_folds(len(lr), form, test):
        RF, RT, mF, mT = lr.iloc[f0:t0], lr.iloc[t0:t1], mkt.iloc[f0:t0], mkt.iloc[t0:t1]
        cols = [t for t in names if RF[t].notna().all() and RT[t].notna().all()]
        CF, betaF, EF = resid_corr(RF[cols], mF)
        CT, _, _ = resid_corr(RT[cols], mT)
        ix = np.array([pos[t] for t in cols]); i, j = np.triu_indices(len(cols), 1)
        F, T, Sv = CF[i, j], CT[i, j], S[ix[i], ix[j]]

        # context: does similarity predict next year's correlation, given this year's?
        z = (Sv - Sv.mean()) / Sv.std()
        A = np.column_stack([np.ones_like(F), F, z])
        coef = np.linalg.lstsq(A, T, rcond=None)[0]
        r2 = lambda X: 1 - ((T - X @ np.linalg.lstsq(X, T, rcond=None)[0]) ** 2).mean() / T.var()

        # matched controls: expected T given the formation-correlation bin, within a similarity group
        b = np.floor((F + 1) / BIN).astype(int)
        def expected(mask):
            g = pd.DataFrame({"b": b[mask], "T": T[mask]}).groupby("b")["T"].agg(["mean", "count"])
            return g["mean"].where(g["count"] >= MIN_CTRL).reindex(b).values
        lo = Sv < np.median(Sv)
        exp_lo, exp_hi = expected(lo), expected(~lo)

        hs = np.where(Sv > np.quantile(Sv, TOP))[0]
        apart = hs[np.argsort(F[hs])[:N_LIST]]
        broad = hs[F[hs] <= np.quantile(F[hs], 0.10)]                 # bottom 10% of the similar pairs: more power
        hc = np.where(F > np.quantile(F, TOP))[0]
        hidden, obvious = hc[np.argsort(Sv[hc])[:N_LIST]], hc[np.argsort(-Sv[hc])[:N_LIST]]
        muF, muT = F[lo].mean(), T[lo].mean()                          # "unrelated pairs" baseline
        keep = lambda G: (T[G].mean() - muT) / (F[G].mean() - muF)     # share of excess correlation that survives

        # catch-up trade: long the formation laggard, short the leader, beta-hedged with formation betas
        cum = (RF[cols].values - np.outer(mF.values, betaF)).sum(0)   # formation hedged return incl. alpha (OLS residuals alone always sum to 0)
        H = (RT[cols].values - np.outer(mT.values, betaF)).sum(0)
        pnl = np.sign(cum[j] - cum[i]) * (H[i] - H[j])

        rows.append({"fold": f"{RF.index[0]:%Y-%m}|{RT.index[0]:%Y-%m}..{RT.index[-1]:%Y-%m}", "stocks": len(cols),
                     "sim_coef": coef[2], "r2_F": r2(A[:, :2]), "r2_F+sim": r2(A),
                     "naive25": np.mean(T[apart] - F[apart]),
                     "excess25": np.nanmean(T[apart] - exp_lo[apart]), "excess_broad": np.nanmean(T[broad] - exp_lo[broad]),
                     "keep_hidden": keep(hidden), "keep_obvious": keep(obvious),
                     "hidden_vs_ctrl": np.nanmean(T[hidden] - exp_hi[hidden]),
                     "trade25": pnl[apart].mean(), "trade_broad": pnl[broad].mean(),
                     "trade_allsim": pnl[hs].mean(), "trade_all": pnl.mean()})
        detail = pd.DataFrame({"a": np.array(cols)[i[apart]], "b": np.array(cols)[j[apart]], "sim": Sv[apart],
                               "F": F[apart], "T": T[apart], "T_ctrl": exp_lo[apart], "trade": pnl[apart]})
    return pd.DataFrame(rows).set_index("fold"), detail

def fm(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return x.mean(), x.mean() / (x.std(ddof=1) / np.sqrt(len(x))), len(x)

res, last = run()
print(res.round(3).to_string())

                          stocks  sim_coef   r2_F  r2_F+sim  naive25  excess25  excess_broad  keep_hidden  keep_obvious  hidden_vs_ctrl  trade25  trade_broad  trade_allsim  trade_all
fold                                                                                                                                                                                  
2017-11|2019-11..2020-11     473     0.009  0.207     0.209    0.070     0.006         0.005        0.580         0.980          -0.119   -0.301       -0.203        -0.058     -0.105
2018-11|2020-11..2021-11     476     0.011  0.570     0.573    0.120     0.080         0.074        0.744         0.982          -0.101    0.053        0.049         0.015      0.016
2019-11|2021-11..2022-11     483     0.027  0.414     0.443    0.258     0.155         0.155        0.393         1.002          -0.239    0.142        0.043         0.028      0.146
2020-11|2022-11..2023-11     486     0.014  0.516     0.526    0.126     0.052       

## 4. Verdict
Fold-level means and t-stats. The critical t is the two-sided 5% value for (folds − 1) degrees of freedom.

In [5]:
def verdict(res):
    k = len(res); crit = stats.t.ppf(0.975, k - 1)
    tab = pd.DataFrame({c: fm(res[c])[:2] for c in res.columns if c != "stocks"}, index=["mean", "t"]).T
    print(tab.round(3).to_string())
    m1, t1, _ = fm(res.excess25)
    m2, t2, _ = fm(res.trade25); m3, t3, _ = fm(res.trade25 - res.trade_allsim)
    print(f"\n{k} folds, critical |t| = {crit:.2f}")
    print(f"H1 information: apart pairs beat matched controls by {m1:+.3f} corr (t {t1:.2f}) -> "
          f"{'PASS' if m1 > 0 and t1 >= crit else 'FAIL'}   [naive rebound {fm(res.naive25)[0]:+.3f}, of which "
          f"{fm(res.naive25)[0] - m1:+.3f} is what matched controls did anyway]")
    print(f"H2 tradeable:   catch-up trade {m2:+.2%}/yr (t {t2:.2f}); vs all similar pairs {m3:+.2%} (t {t3:.2f}) -> "
          f"{'PASS' if m2 > 0 and t2 >= crit and m3 > 0 and t3 >= crit else 'FAIL'}")
    print(f"Hidden twins keep {fm(res.keep_hidden)[0]:.0%} of their excess correlation next year "
          f"(obvious twins {fm(res.keep_obvious)[0]:.0%}); vs similar-business controls {fm(res.hidden_vs_ctrl)[0]:+.3f} (t {fm(res.hidden_vs_ctrl)[1]:.2f})")
    print(f"Context: +1 SD business similarity -> {fm(res.sim_coef)[0]:+.3f} next-year corr (t {fm(res.sim_coef)[1]:.1f}); "
          f"R² of next-year corr: {fm(res.r2_F)[0]:.3f} from this year's alone, {fm(res['r2_F+sim'])[0]:.3f} adding similarity")
verdict(res)

                 mean       t
sim_coef        0.014   4.193
r2_F            0.429  10.002
r2_F+sim        0.438  10.100
naive25         0.138   4.655
excess25        0.080   2.764
excess_broad    0.068   2.903
keep_hidden     0.720   9.266
keep_obvious    0.970  76.442
hidden_vs_ctrl -0.077  -2.069
trade25         0.008   0.117
trade_broad    -0.017  -0.444
trade_allsim   -0.013  -1.004
trade_all       0.007   0.226

7 folds, critical |t| = 2.45
H1 information: apart pairs beat matched controls by +0.080 corr (t 2.76) -> PASS   [naive rebound +0.138, of which +0.058 is what matched controls did anyway]
H2 tradeable:   catch-up trade +0.83%/yr (t 0.12); vs all similar pairs +2.14% (t 0.35) -> FAIL
Hidden twins keep 72% of their excess correlation next year (obvious twins 97%); vs similar-business controls -0.077 (t -2.07)
Context: +1 SD business similarity -> +0.014 next-year corr (t 4.2); R² of next-year corr: 0.429 from this year's alone, 0.438 adding similarity


### Latest fold: the actual apart pairs and what happened next
`T_ctrl` is what dissimilar pairs starting from the same correlation did. `trade` is the hedged laggard-minus-leader log return over the test year.

In [6]:
nm = meta["Security"]
print(last.assign(a_name=nm[last.a].values, b_name=nm[last.b].values).round(3).to_string(index=False))

   a    b   sim      F      T  T_ctrl  trade                  a_name              b_name
 IBM MSFT 0.456 -0.223  0.208  -0.196  0.221                     IBM           Microsoft
STLD   WM 0.469 -0.150  0.003  -0.121  0.114          Steel Dynamics    Waste Management
CTSH MSFT 0.476 -0.145  0.257  -0.121 -0.174               Cognizant           Microsoft
NFLX SKYD 0.540 -0.130  0.249  -0.105 -0.191                 Netflix            Skydance
 CEG   ED 0.482 -0.123 -0.022  -0.105  0.609    Constellation Energy Consolidated Edison
 VST  XEL 0.530 -0.122  0.115  -0.105  0.534            Vistra Corp.         Xcel Energy
 CEG   ES 0.465 -0.115  0.054  -0.085  0.397    Constellation Energy   Eversource Energy
 CEG  XEL 0.516 -0.109  0.204  -0.085 -0.426    Constellation Energy         Xcel Energy
  SO  VST 0.526 -0.106  0.111  -0.085  0.599        Southern Company        Vistra Corp.
 CMS  VST 0.516 -0.102  0.121  -0.085  0.485              CMS Energy        Vistra Corp.
 CEG  CMS 0.474 -0.09

## 5. Robustness: 1-year formation windows
The same test with 1-year formation windows, which gives one more fold. If the verdict flips here, the main result is fragile.

In [7]:
res1, _ = run(form=252, test=252)
verdict(res1)

                 mean       t
sim_coef        0.016   6.854
r2_F            0.415  11.337
r2_F+sim        0.425  11.301
naive25         0.163   6.909
excess25        0.087   4.132
excess_broad    0.073   4.681
keep_hidden     0.679  13.665
keep_obvious    0.970  65.830
hidden_vs_ctrl -0.091  -3.134
trade25        -0.088  -2.702
trade_broad    -0.049  -2.336
trade_allsim   -0.033  -2.716
trade_all      -0.025  -2.236

8 folds, critical |t| = 2.36
H1 information: apart pairs beat matched controls by +0.087 corr (t 4.13) -> PASS   [naive rebound +0.163, of which +0.076 is what matched controls did anyway]
H2 tradeable:   catch-up trade -8.81%/yr (t -2.70); vs all similar pairs -5.50% (t -2.13) -> FAIL
Hidden twins keep 68% of their excess correlation next year (obvious twins 97%); vs similar-business controls -0.091 (t -3.13)
Context: +1 SD business similarity -> +0.016 next-year corr (t 6.9); R² of next-year corr: 0.415 from this year's alone, 0.425 adding similarity
